[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaytAlhikmah/hands-on-llms-for-swes/blob/main/chapters/3/supplemental/entropy_from_counts.ipynb)

# From Counting to Conditional Entropy

**Goal**: Understand how to compute entropy with and without context

We'll follow this exact flow:
1. Show counts, probabilities, and **H(Y)** - entropy WITHOUT context
2. Show counting table → probabilities → **H(Y|X=x)** for each row
3. Explain that to get overall entropy, we weight each row by **P(X=x)**
4. This weighted sum is called **H(Y|X)** - conditional entropy
5. Verify: **H(Y|X) ≤ H(Y)** - context reduces uncertainty

In [ ]:
import csv
import math
import urllib.request

import matplotlib.pyplot as plt
import numpy as np

print('✓ Setup complete!')

## Load Data: Rock-Paper-Scissors

In [ ]:
# Load RPS moves
url = 'https://raw.githubusercontent.com/BaytAlhikmah/hands-on-llms-for-swes/main/chapters/3/rps_opponent_moves.csv'
response = urllib.request.urlopen(url)
lines = response.read().decode('utf-8').strip().split('\n')

moves = []
reader = csv.DictReader(lines)
for row in reader:
    moves.append(row['move'])

short = {'Rock': 'R', 'Paper': 'P', 'Scissors': 'S'}
moves_short = [short[m] for m in moves]

print(f'Total moves: {len(moves)}')
print(f'First 30: {"-".join(moves_short[:30])}')

---

## 1. Entropy WITHOUT Context: H(Y)

**Question**: If we ignore the previous move, how unpredictable is the next move?

We'll:
- Count how often each move appears (as "next move")
- Convert to probabilities P(Y)
- Compute entropy H(Y)

In [ ]:
def entropy(probs):
    """Compute entropy: H = -Σ p·log₂(p)"""
    return -sum(p * math.log2(p) for p in probs if p > 0)

# Setup
rps_labels = ['Rock', 'Paper', 'Scissors']
rps_stoi = {m: i for i, m in enumerate(rps_labels)}

print('='*70)
print('STEP 1: COUNT NEXT MOVES (ignoring previous move)')
print('='*70)
print()

# Count next moves (Y) - ignore what came before
y_counts = [0, 0, 0]
for i in range(len(moves) - 1):
    next_move = moves[i + 1]  # Only care about next move
    y_counts[rps_stoi[next_move]] += 1

total = sum(y_counts)

print('Counts of next moves:')
for i, label in enumerate(rps_labels):
    print(f'  {label}: {y_counts[i]}')
print(f'\nTotal: {total}')

In [ ]:
print('='*70)
print('STEP 2: CONVERT TO PROBABILITIES P(Y)')
print('='*70)
print()

# Convert to probabilities
y_probs = [c / total for c in y_counts]

print('P(Y = next move):')
for i, label in enumerate(rps_labels):
    print(f'  P(Y={label}) = {y_counts[i]}/{total} = {y_probs[i]:.4f}')
print(f'\nSum: {sum(y_probs):.4f} ✓')

In [ ]:
print('='*70)
print('STEP 3: COMPUTE ENTROPY H(Y)')
print('='*70)
print()

H_Y = entropy(y_probs)

print('H(Y) = -Σ P(Y=y) · log₂ P(Y=y)')
print()
for i, label in enumerate(rps_labels):
    p = y_probs[i]
    term = -p * math.log2(p) if p > 0 else 0
    print(f'  -{p:.4f} · log₂({p:.4f}) = {term:.4f}')
print(f'\nH(Y) = {H_Y:.4f} bits')
print()
print(f'→ WITHOUT knowing the previous move, we need {H_Y:.4f} bits on average')
print(f'  to encode the next move')

### Visualize H(Y)

In [ ]:
# Visualize the marginal distribution and entropy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: Probability distribution
colors = ['#e74c3c', '#3498db', '#2ecc71']
bars = ax1.bar(rps_labels, y_probs, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax1.axhline(y=1/3, color='black', linestyle='--', linewidth=1.5, alpha=0.5, label='Uniform (1/3)')
for bar, p in zip(bars, y_probs):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{p:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_ylabel('Probability', fontsize=11)
ax1.set_title('Distribution P(Y)\n(ignoring context)', fontsize=12, fontweight='bold')
ax1.set_ylim([0, max(y_probs) * 1.2])
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

# Right: Show entropy
uniform_h = entropy([1/3, 1/3, 1/3])
ax2.bar(['H(Y)'], [H_Y], color='#e74c3c', alpha=0.7, edgecolor='black', linewidth=2)
ax2.axhline(y=uniform_h, color='black', linestyle='--', linewidth=1.5, 
           label=f'Uniform max = {uniform_h:.3f} bits')
ax2.text(0, H_Y + 0.02, f'{H_Y:.4f} bits', ha='center', va='bottom', 
        fontsize=13, fontweight='bold')
ax2.set_ylabel('Entropy (bits)', fontsize=11)
ax2.set_title('Entropy H(Y)\n(without context)', fontsize=12, fontweight='bold')
ax2.set_ylim([0, uniform_h * 1.2])
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'H(Y) = {H_Y:.4f} bits (marginal entropy)')
print(f'Max possible (uniform) = {uniform_h:.4f} bits')
print(f'\nThe distribution is close to uniform → high unpredictability!')

---

## 2. Entropy WITH Context: Row Entropies H(Y|X=x)

**Question**: If we DO know the previous move, how unpredictable is the next move?

We'll:
- Build a counting table (transitions)
- Normalize each row to get P(Y|X=x)
- Compute H(Y|X=x) for each row

In [ ]:
print('='*70)
print('BUILD TRANSITION COUNTING TABLE')
print('='*70)
print()

# Count transitions: counts[i][j] = how many times j followed i
counts = [[0] * 3 for _ in range(3)]
for i in range(len(moves) - 1):
    prev = rps_stoi[moves[i]]
    curr = rps_stoi[moves[i + 1]]
    counts[prev][curr] += 1

print('Counting table:')
print(f'              {"Rock":>8s} {"Paper":>8s} {"Scissors":>8s}   Total')
print('-'*60)
for i, prev_label in enumerate(rps_labels):
    row_total = sum(counts[i])
    print(f'{prev_label:<10s}    {counts[i][0]:>4d}     {counts[i][1]:>4d}       {counts[i][2]:>4d}      {row_total:>4d}')
print()
print('Each row: how often each move followed that context')

### Visualize Counting Table

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(counts, cmap='Blues', aspect='auto')

for i in range(3):
    for j in range(3):
        ax.text(j, i, counts[i][j], ha="center", va="center", 
               color="black", fontsize=14, fontweight='bold')

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(rps_labels)
ax.set_yticklabels(rps_labels)
ax.set_xlabel('Next move', fontsize=12)
ax.set_ylabel('Previous move (context)', fontsize=12)
ax.set_title('Counting Table\nHow often did Y follow X?', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
print('='*70)
print('NORMALIZE EACH ROW → P(Y|X=x)')
print('='*70)
print()

# Normalize to get conditional probabilities
probs = [[0.0] * 3 for _ in range(3)]

for i, prev_label in enumerate(rps_labels):
    row_total = sum(counts[i])
    print(f'After {prev_label}:')
    print(f'  Counts: {counts[i]}')
    print(f'  Divide by {row_total}:')
    
    for j in range(3):
        probs[i][j] = counts[i][j] / row_total
    
    print(f'  P(Y|X={prev_label}) = [{probs[i][0]:.3f}, {probs[i][1]:.3f}, {probs[i][2]:.3f}]')
    print(f'  Sum = {sum(probs[i]):.3f} ✓')
    print()

### Visualize Probabilities

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(probs, cmap='Purples', aspect='auto', vmin=0, vmax=1)

for i in range(3):
    for j in range(3):
        color = 'white' if probs[i][j] > 0.5 else 'black'
        ax.text(j, i, f'{probs[i][j]:.2f}', ha="center", va="center", 
               color=color, fontsize=13, fontweight='bold')

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(rps_labels)
ax.set_yticklabels(rps_labels)
ax.set_xlabel('Next move', fontsize=12)
ax.set_ylabel('Previous move (context)', fontsize=12)
ax.set_title('Conditional Probabilities P(Y|X=x)\nEach row is a distribution', 
            fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
print('='*70)
print('COMPUTE H(Y|X=x) FOR EACH ROW')
print('='*70)
print()

row_entropies = []

for i, prev_label in enumerate(rps_labels):
    p = probs[i]
    h = entropy(p)
    row_entropies.append(h)
    
    print(f'After {prev_label}:')
    print(f'  P(Y|X={prev_label}) = [{p[0]:.3f}, {p[1]:.3f}, {p[2]:.3f}]')
    print(f'  H(Y|X={prev_label}) = {h:.4f} bits')
    print()

uniform_h = entropy([1/3, 1/3, 1/3])
print(f'Compare to uniform: {uniform_h:.4f} bits')
print()
print('Lower entropy = more predictable when we know that context!')

### Visualize Row Entropies

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(rps_labels, row_entropies, color=colors, alpha=0.7, 
             edgecolor='black', linewidth=2)
ax.axhline(y=uniform_h, color='black', linestyle='--', linewidth=2, 
          label=f'Uniform = {uniform_h:.3f} bits')

for bar, h in zip(bars, row_entropies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
           f'{h:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('Entropy (bits)', fontsize=12)
ax.set_xlabel('Previous move (context)', fontsize=12)
ax.set_title('Row Entropies H(Y|X=x)\nHow predictable is each context?', 
            fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, uniform_h * 1.15])

plt.tight_layout()
plt.show()

print('Each bar = entropy of one row = H(Y|X=x) for that specific context')

---

## 3. Weighting by P(X=x)

**Question**: We have 3 different entropies (one per row). How do we combine them?

**Answer**: Weight each row's entropy by how often that context appears!

- Frequent contexts → higher weight
- Rare contexts → lower weight

In [ ]:
print('='*70)
print('COMPUTE WEIGHTS P(X=x)')
print('='*70)
print()

total_transitions = sum(sum(row) for row in counts)
weights = []

print('How often does each context appear?')
print()
for i, prev_label in enumerate(rps_labels):
    row_total = sum(counts[i])
    weight = row_total / total_transitions
    weights.append(weight)
    print(f'  P(X={prev_label}) = {row_total}/{total_transitions} = {weight:.4f}')

print(f'\nSum of weights: {sum(weights):.4f} ✓')
print()
print('These are our weights for the weighted average!')

In [ ]:
print('='*70)
print('WEIGHTED CONTRIBUTIONS')
print('='*70)
print()

contributions = []

print('For each row: P(X=x) × H(Y|X=x)')
print()
for i, prev_label in enumerate(rps_labels):
    contrib = weights[i] * row_entropies[i]
    contributions.append(contrib)
    print(f'  {prev_label}: {weights[i]:.4f} × {row_entropies[i]:.4f} = {contrib:.4f}')

print(f'\nThese weighted contributions will sum to H(Y|X)!')

### Visualize Weighting

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Row entropies (unweighted)
bars1 = ax1.bar(rps_labels, row_entropies, color=colors, alpha=0.7, 
               edgecolor='black', linewidth=2)
for bar, h in zip(bars1, row_entropies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{h:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax1.set_ylabel('Entropy (bits)', fontsize=11)
ax1.set_xlabel('Context', fontsize=11)
ax1.set_title('H(Y|X=x) per row\n(unweighted)', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim([0, max(row_entropies) * 1.15])

# Right: Weighted contributions
bars2 = ax2.bar(rps_labels, contributions, color=colors, alpha=0.7, 
               edgecolor='black', linewidth=2)
for i, (bar, contrib) in enumerate(zip(bars2, contributions)):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{contrib:.3f}\n(w={weights[i]:.2f})', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_ylabel('Weighted contribution (bits)', fontsize=11)
ax2.set_xlabel('Context', fontsize=11)
ax2.set_title('P(X=x) × H(Y|X=x)\n(weighted by frequency)', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim([0, max(contributions) * 1.25])

fig.text(0.5, 0.95, '→ Weight by P(X=x) →', 
        ha='center', fontsize=14, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print('The right chart shows what each context contributes to the overall entropy')

---

## 4. This Weighted Sum is H(Y|X)

**Definition**: The weighted average of row entropies is called **conditional entropy**

$$H(Y|X) = \sum_{x} P(X=x) \cdot H(Y|X=x)$$

This is the entropy of Y **given** that we know X.

In [ ]:
print('='*70)
print('CONDITIONAL ENTROPY H(Y|X)')
print('='*70)
print()

H_Y_given_X = sum(contributions)

print('H(Y|X) = sum of weighted contributions')
print()
for i, label in enumerate(rps_labels):
    print(f'  + {contributions[i]:.4f}  (from {label})')
print('  ' + '-'*30)
print(f'  = {H_Y_given_X:.4f} bits')
print()
print(f'H(Y|X) = {H_Y_given_X:.4f} bits')
print()
print(f'→ WITH context (knowing previous move), we need {H_Y_given_X:.4f} bits')
print(f'  on average to encode the next move')

### Visualize the Sum

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Stacked bar showing the sum
bottom = 0
for i, label in enumerate(rps_labels):
    ax.bar(['H(Y|X)'], [contributions[i]], bottom=bottom, 
          color=colors[i], alpha=0.7, edgecolor='black', linewidth=2,
          label=f'{label}: {contributions[i]:.3f}')
    # Add text in the middle of each segment
    ax.text(0, bottom + contributions[i]/2, 
           f'{contributions[i]:.3f}',
           ha='center', va='center', fontsize=11, fontweight='bold')
    bottom += contributions[i]

# Add total at top
ax.text(0, bottom + 0.02, f'Total = {H_Y_given_X:.4f} bits',
       ha='center', va='bottom', fontsize=13, fontweight='bold', color='red')

ax.set_ylabel('Entropy (bits)', fontsize=12)
ax.set_title('Conditional Entropy H(Y|X)\nWeighted sum of row entropies', 
            fontsize=13, fontweight='bold')
ax.legend(fontsize=11, title='Contributions:', title_fontsize=11)
ax.set_ylim([0, bottom * 1.1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'H(Y|X) = {H_Y_given_X:.4f} bits (conditional entropy)')

---

## 5. Verify: H(Y|X) ≤ H(Y)

**Property from Chapter 2**: Conditioning reduces entropy

Knowing X can only reduce (or leave unchanged) uncertainty about Y.

In [ ]:
print('='*70)
print('COMPARE: H(Y) vs H(Y|X)')
print('='*70)
print()

print(f'H(Y)   = {H_Y:.4f} bits   (WITHOUT context)')
print(f'H(Y|X) = {H_Y_given_X:.4f} bits   (WITH context)')
print()

reduction = H_Y - H_Y_given_X
reduction_pct = (reduction / H_Y) * 100

print(f'Reduction: {reduction:.4f} bits ({reduction_pct:.1f}%)')
print()
print(f'✓ Verified: H(Y|X) ≤ H(Y)')
print(f'  {H_Y_given_X:.4f} ≤ {H_Y:.4f}')
print()
print('Context reduces uncertainty!')
print(f'Knowing the previous move saves us {reduction:.4f} bits on average.')

### Visualize the Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

categories = ['H(Y)\n(no context)', 'H(Y|X)\n(with context)']
values = [H_Y, H_Y_given_X]
bar_colors = ['#e74c3c', '#2ecc71']

bars = ax.bar(categories, values, color=bar_colors, alpha=0.7, 
             edgecolor='black', linewidth=2, width=0.5)

# Add value labels
for bar, val in zip(bars, values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
           f'{val:.4f} bits', ha='center', va='bottom', 
           fontsize=13, fontweight='bold')

# Show reduction with arrow
ax.annotate('', xy=(1, H_Y_given_X), xytext=(1, H_Y),
           arrowprops=dict(arrowstyle='<->', color='red', lw=3))
ax.text(1.2, (H_Y + H_Y_given_X) / 2, 
       f'Reduction\n{reduction:.4f} bits\n({reduction_pct:.1f}%)',
       fontsize=12, fontweight='bold', color='red', va='center',
       bbox=dict(boxstyle='round', facecolor='white', edgecolor='red', linewidth=2))

ax.set_ylabel('Entropy (bits)', fontsize=12)
ax.set_title('Property: Conditioning Reduces Entropy\nH(Y|X) ≤ H(Y)', 
            fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, H_Y * 1.25])

plt.tight_layout()
plt.show()

---

## Summary

We showed the complete pipeline from data to conditional entropy:

### 1. **H(Y)** - Marginal Entropy
- Count next moves (ignoring context)
- Normalize: P(Y)
- Compute: H(Y) = -Σ P(Y=y) log₂ P(Y=y)
- **Result**: H(Y) = {H_Y:.4f} bits

### 2. **H(Y|X=x)** - Row Entropies  
- Build transition counting table
- Normalize each row: P(Y|X=x)
- Compute entropy per row
- **Result**: 3 different entropies (one per context)

### 3. **Weighting by P(X=x)**
- Compute how often each context appears
- Weight each row entropy: P(X=x) × H(Y|X=x)

### 4. **H(Y|X)** - Conditional Entropy
- Sum the weighted contributions
- **Formula**: H(Y|X) = Σ P(X=x) · H(Y|X=x)
- **Result**: H(Y|X) = {H_Y_given_X:.4f} bits

### 5. **Verification**
- H(Y|X) ≤ H(Y) ✓
- Context reduces entropy by {reduction:.4f} bits ({reduction_pct:.1f}%)

**Key insight**: Knowing the previous move makes the next move more predictable!